In [5]:
!pip install --upgrade --no-cache-dir \
  --root-user-action=ignore \
  --trusted-host network-service \
  --index-url http://network-service:8080/sdk/simple \
  cisei-planning-sdk


Looking in indexes: http://network-service:8080/sdk/simple


In [4]:
import requests

url = "http://planning-service:8080/planning-sdk-0/set_link"
params = {
    "tx_lat": -25.423532434217368,
    "tx_lon": -49.29399235240344,
    "rx_lat": -25.425356577135183,
    "rx_lon": -49.29212222420152,
    "tx_ha": 100.0,
    "rx_ha": 7.0,
    "freq_mhz": 915.0,
}

r = requests.get(url, params=params, timeout=120)
print(r.status_code)
print(r.text[:2000])

401
{"detail":"X-Token header missing"}


In [6]:
from cisei_planning_sdk import PlanningClient, __version__

print(__version__)

client = PlanningClient("http://network-service:8080", 
                        user_id="my_notebook",
                        timeout=900)

scenario = client.define_scenario(
    name="sdk_test",
    planner="cell",
    scenario="scenario_exports/cell_planner_sector_antenna.toml",
)

candidate_edges = scenario.build_candidate_edges()
candidate_edges["counts"]


0.1.9


{'sites': 13,
 'rpl_nodes': 16,
 'candidate_edges': 21,
 'metric_edges': 0,
 'planned_nodes': 0,
 'planned_edges': 0}

In [7]:
metrics = scenario.compute_metrics(
    geo_timeout=300,
    include_features=False
)

In [10]:
solution = scenario.solve()

solution["counts"]
solution["planned_edges"][:5]

[{'src': 'copel:d0:i0', 'dst': 'torre:d0:i2', 'metric': 1.0},
 {'src': 'tenis:d0:i0', 'dst': 'torre:d0:i1', 'metric': 1.0},
 {'src': 'praca_29:d0:i0', 'dst': 'torre:d0:i1', 'metric': 1.0},
 {'src': 'torre:d0:i1', 'dst': 'guaira:d0:i0', 'metric': 4.424564720781856},
 {'src': 'torre:d0:i2', 'dst': 'woodland:d0:i0', 'metric': 10.425350820044244}]

In [11]:
evaluation = scenario.evaluate(
    rank_threshold=10,
    primary_tech="lte",
)
evaluation.keys()
evaluation["cell"].keys()
evaluation["cell"]["client_service"][:3]
evaluation["cell"]["cell_load"]

[{'cell_site_id': 'itallia',
  'cell_node_id': 'itallia:d0:i0',
  'cell_tech': 'lte',
  'antenna_id': 'lte_sector_120',
  'freq_mhz': 915.0,
  'served_clients': 0},
 {'cell_site_id': 'itallia',
  'cell_node_id': 'itallia:d0:i1',
  'cell_tech': 'lte',
  'antenna_id': 'lte_sector_240',
  'freq_mhz': 915.0,
  'served_clients': 0},
 {'cell_site_id': 'torre',
  'cell_node_id': 'torre:d0:i0',
  'cell_tech': 'lte',
  'antenna_id': 'lte_sector_0',
  'freq_mhz': 915.0,
  'served_clients': 0},
 {'cell_site_id': 'torre',
  'cell_node_id': 'torre:d0:i1',
  'cell_tech': 'lte',
  'antenna_id': 'lte_sector_120',
  'freq_mhz': 915.0,
  'served_clients': 3},
 {'cell_site_id': 'torre',
  'cell_node_id': 'torre:d0:i2',
  'cell_tech': 'lte',
  'antenna_id': 'lte_sector_240',
  'freq_mhz': 915.0,
  'served_clients': 8}]

In [13]:
result = scenario.export_result(
    rank_threshold=10,
    primary_tech="lte",
)



In [14]:
evaluation_from_result = client.evaluate_result(
    result,
    rank_threshold=10,
    primary_tech="lte",
    solution_kind="cell",
)


In [15]:

evaluation_from_result["cell"]["cell_load"]

[{'cell_site_id': 'itallia',
  'cell_node_id': 'itallia:d0:i0',
  'cell_tech': 'lte',
  'antenna_id': 'lte_sector_120',
  'freq_mhz': 915.0,
  'served_clients': 0},
 {'cell_site_id': 'itallia',
  'cell_node_id': 'itallia:d0:i1',
  'cell_tech': 'lte',
  'antenna_id': 'lte_sector_240',
  'freq_mhz': 915.0,
  'served_clients': 0},
 {'cell_site_id': 'torre',
  'cell_node_id': 'torre:d0:i0',
  'cell_tech': 'lte',
  'antenna_id': 'lte_sector_0',
  'freq_mhz': 915.0,
  'served_clients': 0},
 {'cell_site_id': 'torre',
  'cell_node_id': 'torre:d0:i1',
  'cell_tech': 'lte',
  'antenna_id': 'lte_sector_120',
  'freq_mhz': 915.0,
  'served_clients': 3},
 {'cell_site_id': 'torre',
  'cell_node_id': 'torre:d0:i2',
  'cell_tech': 'lte',
  'antenna_id': 'lte_sector_240',
  'freq_mhz': 915.0,
  'served_clients': 8}]

In [16]:
evaluation_from_result["quality_summary"]
evaluation_from_result["cell"]["cell_load"]
evaluation_from_result["cell"]["client_service"]
evaluation_from_result["result_edges"]

[{'src': 'copel:d0:i0',
  'dst': 'torre:d0:i2',
  'kind': 'rf',
  'rule': 'cell_primary',
  'src_site': 'copel',
  'dst_site': 'torre',
  'src_tech': 'lte',
  'dst_tech': 'lte',
  'src_freq_mhz': 915.0,
  'dst_freq_mhz': 915.0,
  'src_max_links': None,
  'dst_max_links': None,
  'src_mount_height_m': 7.0,
  'dst_mount_height_m': 100.0,
  'metric': 1.0},
 {'src': 'tenis:d0:i0',
  'dst': 'torre:d0:i1',
  'kind': 'rf',
  'rule': 'cell_primary',
  'src_site': 'tenis',
  'dst_site': 'torre',
  'src_tech': 'lte',
  'dst_tech': 'lte',
  'src_freq_mhz': 915.0,
  'dst_freq_mhz': 915.0,
  'src_max_links': None,
  'dst_max_links': None,
  'src_mount_height_m': 7.0,
  'dst_mount_height_m': 100.0,
  'metric': 1.0},
 {'src': 'praca_29:d0:i0',
  'dst': 'torre:d0:i1',
  'kind': 'rf',
  'rule': 'cell_primary',
  'src_site': 'praca_29',
  'dst_site': 'torre',
  'src_tech': 'lte',
  'dst_tech': 'lte',
  'src_freq_mhz': 915.0,
  'dst_freq_mhz': 915.0,
  'src_max_links': None,
  'dst_max_links': None,
  's

In [17]:
import pandas as pd

pd.DataFrame(evaluation_from_result["cell"]["cell_load"])
pd.DataFrame(evaluation_from_result["cell"]["client_service"])

,client_site_id,client_node_id,client_tech,served,cell_node_id,cell_site_id,cell_tech,selected_metric,rank
0,barigui,barigui:d0:i0,lte,True,torre:d0:i2,torre,lte,1.000000,1.000000
1,cemiterio,cemiterio:d0:i0,lte,True,torre:d0:i2,torre,lte,1.000000,1.000000
2,copel,copel:d0:i0,lte,True,torre:d0:i2,torre,lte,1.000000,1.000000
3,decatlon,decatlon:d0:i0,lte,True,torre:d0:i2,torre,lte,1.468871,1.468871
4,deneka,deneka:d0:i0,lte,True,torre:d0:i2,torre,lte,1.868147,1.868147
5,guaira,guaira:d0:i0,lte,True,torre:d0:i1,torre,lte,4.424565,4.424565
6,praca_29,praca_29:d0:i0,lte,True,torre:d0:i1,torre,lte,1.000000,1.000000
7,saturno,saturno:d0:i0,lte,True,torre:d0:i2,torre,lte,2.604778,2.604778
8,shopping,shopping:d0:i0,lte,True,torre:d0:i2,torre,lte,1.493932,1.493932
9,tenis,tenis:d0:i0,lte,True,torre:d0:i1,torre,lte,1.000000,1.000000
